# 🌸 Lily LTX 2B Distilled — Fast Image → Video

Run this one cell with **Internet ON** and a Kaggle **GPU** attached.

Pipeline: **image → LTX 2B distilled → short low-res native clip → cheap upscale + FPS conversion → MP4**.


In [ ]:
import urllib.request, pathlib, shutil
shutil.rmtree('/kaggle/working/LTX-Video', ignore_errors=True)
URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/main/LILY_LTX2B_FAST_I2V_STUDIO.py'
DST = pathlib.Path('/kaggle/working/LILY_LTX2B_FAST_I2V_STUDIO.py')
urllib.request.urlretrieve(URL, DST)
code = DST.read_text(encoding='utf-8')
# Xet support for current Hugging Face large-file storage.
code = code.replace('\"huggingface-hub==0.30.2\",', '\"huggingface-hub==0.30.2\",\n    \"hf-xet>=1.1.5\",')
# Avoid the brittle PixArt subfolder resolver entirely: snapshot the exact T5/tokenizer files locally first.
code = code.replace('from huggingface_hub import hf_hub_download', 'from huggingface_hub import hf_hub_download, snapshot_download')
old = '''print("📝 Loading PixArt T5 text encoder in CPU RAM (FP16 storage)...")
TOKENIZER = T5Tokenizer.from_pretrained(TEXT_REPO, subfolder="tokenizer")
TEXT_ENCODER = T5EncoderModel.from_pretrained(
    TEXT_REPO,
    subfolder="text_encoder",
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
).cpu().eval()'''
new = '''print("📝 Downloading T5/tokenizer files explicitly (this is the big ~19 GB first-run download)...")
TEXT_SNAPSHOT = snapshot_download(
    repo_id=TEXT_REPO,
    allow_patterns=["tokenizer/*", "text_encoder/*"],
    local_dir="/kaggle/working/pixart_text",
)
TOKENIZER = T5Tokenizer.from_pretrained(str(Path(TEXT_SNAPSHOT) / "tokenizer"), local_files_only=True)
TEXT_ENCODER = T5EncoderModel.from_pretrained(
    str(Path(TEXT_SNAPSHOT) / "text_encoder"),
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    local_files_only=True,
).cpu().eval()'''
if old not in code:
    raise RuntimeError('Launcher patch could not find the T5 loading block; stop rather than running broken code.')
code = code.replace(old, new)
DST.write_text(code, encoding='utf-8')
print('✅ Studio code downloaded + robust local T5 snapshot patch applied:', DST)
exec(compile(code, str(DST), 'exec'))
